In [1]:
import io
import re
import requests
import pandas as pd

URL = "https://www.iposcoop.com/ipos-recently-filed/"
HEADERS = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0 Safari/537.36")}

resp = requests.get(URL, headers=HEADERS, timeout=30)
resp.raise_for_status()
print("status:", resp.status_code, "len:", len(resp.text))

tables = pd.read_html(io.StringIO(resp.text))
print("tables found:", len(tables))
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:6])

status: 200 len: 332953
tables found: 1
0 (500, 10) ['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low']


In [2]:
raw = tables[0].copy()
print(list(raw.columns))
raw.head()

['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-10-22,Aspen Neuroscience,ASPE,Leerink Partners/Piper Sandler/Stifel/UBS Inve...,0.00,NaN,NaN,$100.00,TBA,S/O
1,2026-10-22,Davion Healthcare (Direct Listing),DAVI,Revere Securities (Financial Advisor),25.00,$12.00,$12.00,$300.00,Priced,S/O
2,2026-10-21,Georgia Banking Co. (Direct Listing),GBC,Performance Trust Capital Parnters (Financial ...,7.04,$30.00,$30.00,$211.20,TBA,S/O
3,2026-10-21,Helio Corp. (NASDAQ Uplisting),HLEO,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,TBA,S/O
4,2026-10-21,Rezera Inc.,RZRA,Jefferies/TD Cowen/Cantor/Wedbush PacGrow,0.00,NaN,NaN,$100.00,TBA,S/O


In [3]:
trade_col = [c for c in raw.columns if "expected" in str(c).lower()][0]
print("status column:", trade_col)
print(raw[trade_col].value_counts())

wd = raw[raw[trade_col].astype(str).str.strip().str.lower() == "withdrawn"].copy()
print("\nwithdrawn rows:", len(wd))
wd.head()

status column: Expected To Trade
Expected To Trade
Priced       253
TBA          191
Withdrawn     36
Week of        6
Postponed      5
Wednesday      4
Tuesday        2
Thursday       2
Friday         1
Name: count, dtype: int64

withdrawn rows: 36


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
13,2026-10-19,Kepuni Holdings Inc. (Withdrawn),KPN,Boustead Securities,4.20,$6.00,$6.00,$25.20,Withdrawn,S/O
16,2026-10-16,Klook Technology Limited (Withdrawn),KLK,Goldman Sachs/J.P. Morgan/Morgan Stanley/Citi/...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
20,2026-10-14,"BioVentrix, Inc. (Withdrawn)",BVXX,The Benchmark Company,0.00,NaN,NaN,$17.00,Withdrawn,S/O
43,2026-10-07,Siyata PTT (Withdrawn – Direct Listing),PTT,Digital Offering LLC (Financial Advisor),6.11,$7.00,$7.00,$42.77,Withdrawn,S/O
69,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.00,$15.00,$18.00,$825.00,Withdrawn,S/O


In [4]:
def classify(name: str) -> str:
    n = str(name)
    if "Technologies" in n:
        return "Technologies"
    if ("Acquisition Corp" in n or "Acquisition Corporation" in n or "Corp" in n):
        return "Acquisition Corp"
    if "Inc" in n or "Incorporated" in n:
        return "Inc."
    if "Group" in n:
        return "Group"
    if "Ltd" in n or "Limited" in n:
        return "Limited"
    if "Holdings" in n or "Holding" in n:
        return "Holdings"
    return "Other"

name_col = [c for c in wd.columns if "company" in str(c).lower()][0]
wd["Company Type"] = wd[name_col].apply(classify)
wd[[name_col, "Company Type"]].to_string()

'                                             Company      Company Type\n13                  Kepuni Holdings Inc. (Withdrawn)              Inc.\n16              Klook Technology Limited (Withdrawn)           Limited\n20                      BioVentrix, Inc. (Withdrawn)              Inc.\n43           Siyata PTT (Withdrawn – Direct Listing)             Other\n69                        Holtec Nuclear (Withdrawn)             Other\n84                      Rise Smart Group (Withdrawn)             Group\n86                      C2 Capital Group (Withdrawn)             Group\n94                            OTSAW Ltd. (Withdrawn)           Limited\n97                   Motive Technologies (Withdrawn)      Technologies\n100                    Idea Tech Holding (Withdrawn)          Holdings\n106           Hornbeck Offshore Services (Withdrawn)             Other\n108            Coolbit Technologies Ltd. (Withdrawn)      Technologies\n115        Timber Road Acquisition Corp. (Withdrawn)  Acquisiti

In [5]:
wd.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
13,2026-10-19,Kepuni Holdings Inc. (Withdrawn),KPN,Boustead Securities,4.20,$6.00,$6.00,$25.20,Withdrawn,S/O,Inc.
16,2026-10-16,Klook Technology Limited (Withdrawn),KLK,Goldman Sachs/J.P. Morgan/Morgan Stanley/Citi/...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Limited
20,2026-10-14,"BioVentrix, Inc. (Withdrawn)",BVXX,The Benchmark Company,0.00,NaN,NaN,$17.00,Withdrawn,S/O,Inc.
43,2026-10-07,Siyata PTT (Withdrawn – Direct Listing),PTT,Digital Offering LLC (Financial Advisor),6.11,$7.00,$7.00,$42.77,Withdrawn,S/O,Other
69,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.00,$15.00,$18.00,$825.00,Withdrawn,S/O,Other


In [6]:
def to_num(x):
    """'$8.00' -> 8.0 ; '-', '', NaN -> None"""
    if pd.isna(x):
        return None
    s = str(x).replace("$", "").replace(",", "").strip()
    if s in ("", "-", "–", "—", "N/A", "n/a"):
        return None
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group()) if m else None

low_col  = [c for c in wd.columns if "low"  in str(c).lower()][0]
high_col = [c for c in wd.columns if "high" in str(c).lower()][0]

wd["price_low_num"]  = wd[low_col].apply(to_num)
wd["price_high_num"] = wd[high_col].apply(to_num)
wd["Avg_price"] = wd[["price_low_num", "price_high_num"]].mean(axis=1)

print("missing Avg_price:", wd["Avg_price"].isna().sum())
wd[[low_col, high_col, "Avg_price"]].head(10)

missing Avg_price: 8


,Price Low,Price High,Avg_price
13,$6.00,$6.00,6.0
16,NaN,NaN,NaN
20,NaN,NaN,NaN
43,$7.00,$7.00,7.0
69,$15.00,$18.00,16.5
84,$4.00,$4.00,4.0
86,$4.00,$5.00,4.5
94,$4.50,$5.50,5.0
97,NaN,NaN,NaN
100,$4.00,$5.00,4.5


In [7]:
shares_col = [c for c in wd.columns if "shares" in str(c).lower()][0]
vol_col    = [c for c in wd.columns if "vol"    in str(c).lower()][0]
print("shares:", shares_col, "| vol:", vol_col)

wd["Shares_num"] = wd[shares_col].apply(to_num)
wd["Est_vol_num"] = wd[vol_col].apply(to_num)

print("missing shares:", wd["Shares_num"].isna().sum(),
      "| missing vol:", wd["Est_vol_num"].isna().sum())
wd[[shares_col, "Shares_num", vol_col, "Est_vol_num"]].head(10)

shares: Shares (millions) | vol: Est $ Vol (millions)
missing shares: 0 | missing vol: 0


,Shares (millions),Shares_num,Est $ Vol (millions),Est_vol_num
13,4.20,4.20,$25.20,25.20
16,0.00,0.00,$100.00,100.00
20,0.00,0.00,$17.00,17.00
43,6.11,6.11,$42.77,42.77
69,50.00,50.00,$825.00,825.00
84,1.80,1.80,$7.20,7.20
86,3.75,3.75,$16.88,16.88
94,4.40,4.40,$22.00,22.00
97,0.00,0.00,$100.00,100.00
100,2.00,2.00,$9.00,9.00


In [8]:
zero_shares = wd[wd["Shares_num"] == 0]
print("rows with 0 shares:", len(zero_shares))
zero_shares[[name_col, "Shares_num", "Avg_price", "Est_vol_num"]]

rows with 0 shares: 8


,Company,Shares_num,Avg_price,Est_vol_num
16,Klook Technology Limited (Withdrawn),0.0,NaN,100.0
20,"BioVentrix, Inc. (Withdrawn)",0.0,NaN,17.0
97,Motive Technologies (Withdrawn),0.0,NaN,100.0
106,Hornbeck Offshore Services (Withdrawn),0.0,NaN,100.0
231,Kilcoy Global Foods Ltd. (Withdrawn),0.0,NaN,100.0
318,Xinxu Copper Industry Technology Ltd.,0.0,NaN,15.0
377,Going International Holding Company Limited,0.0,NaN,20.0
413,Invea Therapeutics (Withdrawn),0.0,NaN,75.0


In [9]:
computed = wd["Shares_num"] * wd["Avg_price"]
wd["Shares_offered_value"] = computed.fillna(wd["Est_vol_num"])

print("rows using computed value:", computed.notna().sum())
print("rows falling back to Est $ Vol:", computed.isna().sum())
print("still null after fallback:", wd["Shares_offered_value"].isna().sum())
wd[[name_col, "Company Type", "Shares_num", "Avg_price",
    "Est_vol_num", "Shares_offered_value"]].to_string()

rows using computed value: 28
rows falling back to Est $ Vol: 8
still null after fallback: 0


'                                             Company      Company Type  Shares_num  Avg_price  Est_vol_num  Shares_offered_value\n13                  Kepuni Holdings Inc. (Withdrawn)              Inc.        4.20       6.00        25.20               25.2000\n16              Klook Technology Limited (Withdrawn)           Limited        0.00        NaN       100.00              100.0000\n20                      BioVentrix, Inc. (Withdrawn)              Inc.        0.00        NaN        17.00               17.0000\n43           Siyata PTT (Withdrawn – Direct Listing)             Other        6.11       7.00        42.77               42.7700\n69                        Holtec Nuclear (Withdrawn)             Other       50.00      16.50       825.00              825.0000\n84                      Rise Smart Group (Withdrawn)             Group        1.80       4.00         7.20                7.2000\n86                      C2 Capital Group (Withdrawn)             Group        3.75       

In [10]:
agg = (wd.groupby("Company Type")["Shares_offered_value"]
         .sum()
         .sort_values(ascending=False))
print(agg.round(2).to_string())

top = agg.idxmax()
print(f"\nHighest: {top} — ${agg.max():,.2f} million")

Company Type
Other               1152.71
Acquisition Corp     485.00
Inc.                 393.20
Limited              313.25
Holdings             311.66
Technologies         184.90
Group                 56.58

Highest: Other — $1,152.71 million


In [11]:
wd["File Date"] = pd.to_datetime(wd["File Date"], errors="coerce")
sub = wd[wd["File Date"] < "2026-09-11"]
print("rows:", len(sub))
print(sub.groupby("Company Type")["Shares_offered_value"].sum()
         .sort_values(ascending=False).round(2).to_string())

rows: 28
Company Type
Acquisition Corp    485.00
Inc.                351.00
Holdings            311.66
Other               284.94
Limited             191.25
Technologies        184.90
Group                32.50


In [15]:
import sys
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install yfinance pandas numpy requests lxml

Looking in links: /var/folders/6r/qn2nw9l17zs4b7s1wnb0v8q40000gn/T/tmpjgrzfuhy
Processing /var/folders/6r/qn2nw9l17zs4b7s1wnb0v8q40000gn/T/tmpjgrzfuhy/pip-25.3-py3-none-any.whl
  Using cached yfinance-1.7.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.16.3-cp310-abi3-macosx_10_9_x86_64.whl.metadata (17 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
Using cached yfinance-1.7.0-py3-none-any.whl (149 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached curl_cffi-0.16.3-cp310-abi3-macosx_10_9_x86_64.whl (3.0 MB)
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [yfinance]/11 [yfinance]]]

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


In [16]:
# Question 2: IPOs - Median Sharpe Ratio for 2025 Pricings

import io, re
import numpy as np
import pandas as pd
import requests
import yfinance as yf

URL = "https://www.iposcoop.com/2025-pricings/"
HEADERS = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0 Safari/537.36")}

resp = requests.get(URL, headers=HEADERS, timeout=30)
resp.raise_for_status()
tables = pd.read_html(io.StringIO(resp.text))
print("tables:", len(tables))
ipos = tables[0].copy()
print(ipos.shape)
print(list(ipos.columns))
ipos.head()

tables: 1
(231, 10)
['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price', 'Return', 'SCOOP Rating']


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


In [17]:
date_col   = [c for c in ipos.columns if "offer" in str(c).lower() and "date" in str(c).lower()][0]
return_col = [c for c in ipos.columns if "return" in str(c).lower()][0]
sym_col    = [c for c in ipos.columns if "symbol" in str(c).lower()][0]
print("using:", date_col, "|", return_col, "|", sym_col)

ipos[date_col] = pd.to_datetime(ipos[date_col], errors="coerce")

def pct_to_num(x):
    if pd.isna(x): return np.nan
    s = str(x).replace("%", "").replace(",", "").strip()
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group()) if m else np.nan

ipos["return_num"] = ipos[return_col].apply(pct_to_num)

sel = ipos[(ipos[date_col] < "2025-09-01") & (ipos["return_num"] != 0)].copy()
print("filtered:", len(sel))
sel[[date_col, sym_col, return_col, "return_num"]].head()

using: Offer Date | Return | Symbol
filtered: 146


,Offer Date,Symbol,Return,return_num
68,2025-08-29,PMI,-58.00%,-58.00
69,2025-08-28,TGHL,-90.00%,-90.00
70,2025-08-28,THH,636.75%,636.75
71,2025-08-26,CURX,-90.75%,-90.75
72,2025-08-21,CEPF,1.20%,1.20


In [24]:
tickers = (sel[sym_col].astype(str).str.strip().str.upper()
             .str.replace(r"[^A-Z.\-]", "", regex=True))
tickers = sorted(t for t in tickers.unique() if t and t != "NAN")
print("tickers to fetch:", len(tickers))

raw = yf.download(tickers, start="2024-12-01", end="2026-09-12",
                  group_by="ticker", auto_adjust=True,
                  progress=False, threads=True)
print("downloaded block:", raw.shape)

tickers to fetch: 146


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SDM"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WGRX"}}}
$SKBL: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$MJID: possibly delisted; no timezone found
$EPWK: possibly delisted; no price data found  (1d 2024-12-01 -> 2026-09-12)
$MCTR: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found

14 Failed downloads:
['SKBL', 'SDM', 'WGRX', 'AHL', 'TBH', 'CAEP', 'CEPT', 'MTSR', 'PTNM', 'MJID', 'M

downloaded block: (445, 744)


In [25]:
available = set(raw.columns.get_level_values(0))
frames, missing = [], []

for t in tickers:
    if t not in available:
        missing.append(t); continue
    sub = raw[t].dropna(how="all").copy()
    if sub.empty or sub["Close"].notna().sum() == 0:
        missing.append(t); continue
    sub["Ticker"] = t
    frames.append(sub)

stocks_df = pd.concat(frames).sort_index().reset_index()
print("stocks with data:", stocks_df['Ticker'].nunique())
print("no data (likely delisted):", len(missing))
print(missing)

stocks with data: 132
no data (likely delisted): 14
['AGH', 'AHL', 'CAEP', 'CEPT', 'EMPG', 'EPWK', 'MCTR', 'MJID', 'MTSR', 'PTNM', 'SDM', 'SKBL', 'TBH', 'WGRX']


In [26]:
g = stocks_df.groupby("Ticker")["Close"]

stocks_df["growth_252d"] = g.transform(lambda s: s / s.shift(252))
stocks_df["volatility"]  = g.transform(lambda s: s.rolling(30).std() * np.sqrt(252))
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

stocks_df[["Date", "Ticker", "Close", "growth_252d", "volatility", "Sharpe"]].tail()

Price,Date,Ticker,Close,growth_252d,volatility,Sharpe
44286,2026-09-11,FLOC,20.709999,1.325809,16.347653,0.078042
44287,2026-09-11,FLY,21.129999,0.494038,35.586739,0.012478
44288,2026-09-11,FMFC,7.920000,0.018316,40.041010,-0.000791
44289,2026-09-11,ETOR,30.350000,0.711940,43.267292,0.015299
44290,2026-09-11,ZYBT,1.550000,0.149038,3.734725,0.026518


In [27]:
one = stocks_df[stocks_df["Ticker"] == stocks_df["Ticker"].iloc[0]]
print(one[["Date","Close","growth_252d"]].head(3))   # growth should be NaN early
print(one[["Date","Close","growth_252d"]].tail(3))

Price       Date  Close  growth_252d
0     2024-12-02  280.0          NaN
8     2024-12-03  280.0          NaN
10    2024-12-04  280.0          NaN
Price       Date  Close  growth_252d
43926 2026-09-09   2.81     0.061758
44135 2026-09-10   2.78     0.050463
44223 2026-09-11   2.75     0.043699


In [29]:
day = stocks_df[stocks_df["Date"] == "2026-09-11"]
print("rows on 2026-09-11:", len(day))

desc = day[["growth_252d", "volatility", "Sharpe"]].describe()
print(desc.round(4).to_string())

print(f"\nMedian Sharpe: {day['Sharpe'].median():.4f}")
print(f"Median growth_252d: {day['growth_252d'].median():.4f}  "
      f"| Mean: {day['growth_252d'].mean():.4f}")
print(f"Reached 252-day milestone: {day['growth_252d'].notna().sum()}")

rows on 2026-09-11: 132
Price  growth_252d  volatility    Sharpe
count     128.0000    131.0000  128.0000
mean        1.0194     32.5439       inf
std         3.0336     76.3254       NaN
min         0.0010      0.0000   -0.0401
25%         0.1340      2.4484    0.0090
50%         0.5945      8.3802    0.0459
75%         1.0360     24.8108    0.1250
max        33.6383    627.6981       inf

Median Sharpe: 0.0459
Median growth_252d: 0.5945  | Mean: 1.0194
Reached 252-day milestone: 128


/Users/osiosman/stock-analytics-zoomcamp/.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [30]:
# Question 3 - What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?

g = stocks_df.groupby("Ticker")["Close"]

for m in range(1, 13):
    stocks_df[f"future_growth_{m}_m"] = g.transform(
        lambda s, k=21 * m: s.shift(-k) / s
    )

cols = [f"future_growth_{m}_m" for m in range(1, 13)]
stocks_df[["Date", "Ticker", "Close"] + cols[:3]].head()

Price,Date,Ticker,Close,future_growth_1_m,future_growth_2_m,future_growth_3_m
0,2024-12-02,CIIT,280.000000,1.000000,1.000000,1.000000
1,2024-12-02,CAPS,3.200000,0.859375,0.781250,1.218750
2,2024-12-02,AVBH,22.110001,0.985979,1.138399,1.047037
3,2024-12-02,ALM,1.469250,1.056662,1.509954,2.052068
4,2024-12-02,PPCB,0.015000,1.333333,16666.667039,11666.666927


In [31]:
first = stocks_df.groupby("Ticker")["Date"].min().reset_index()

offers = sel[[sym_col, date_col]].copy()
offers.columns = ["Ticker", "offer_date"]
offers["Ticker"] = offers["Ticker"].astype(str).str.strip().str.upper()

chk = first.merge(offers, on="Ticker", how="left")
chk["gap_days"] = (chk["Date"] - chk["offer_date"]).dt.days
print(chk["gap_days"].describe())
print("\nmin_date well before offer date:")
print(chk[chk["gap_days"] < -5].sort_values("gap_days").to_string())

count    132.000000
mean       4.583333
std       82.465634
min     -256.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      553.000000
Name: gap_days, dtype: float64

min_date well before offer date:
    Ticker       Date offer_date  gap_days
100   PPCB 2024-12-02 2025-08-15      -256
13    AVBH 2024-12-02 2025-08-08      -249
5      ALM 2024-12-02 2025-07-14      -224
28    CIIT 2024-12-02 2025-04-10      -129
21    CAPS 2024-12-02 2025-03-06       -94


In [32]:
print(chk[chk["gap_days"] > 5].sort_values("gap_days", ascending=False).to_string())

    Ticker       Date offer_date  gap_days
83    MIMI 2026-07-17 2025-01-10       553
27    CIGL 2026-08-04 2025-04-22       469
74    MAMK 2026-07-17 2025-07-07       375
117   TOPW 2025-09-04 2025-04-02       155


In [33]:
bad = chk.loc[chk["gap_days"].abs() > 5, "Ticker"].tolist()
print("excluding", len(bad), ":", bad)
first = stocks_df.groupby("Ticker")["Date"].min().reset_index()
first = first[~first["Ticker"].isin(bad)]
print("tickers kept:", len(first))

excluding 9 : ['ALM', 'AVBH', 'CAPS', 'CIGL', 'CIIT', 'MAMK', 'MIMI', 'PPCB', 'TOPW']
tickers kept: 123


In [34]:
entry = stocks_df.merge(first, on=["Ticker", "Date"], how="inner")
print("entry rows:", len(entry))
entry[["Ticker", "Date", "Close"] + cols[:3]].head()

entry rows: 123


,Ticker,Date,Close,future_growth_1_m,future_growth_2_m,future_growth_3_m
0,INLF,2025-01-02,14240.00,0.982022,0.408989,0.332584
1,ZYBT,2025-01-07,4.86,0.866255,0.903704,0.827160
2,CEPO,2025-01-07,10.05,1.005970,1.006965,1.024876
3,MASK,2025-01-08,81.25,0.812308,0.867692,0.972308
4,UFG,2025-01-14,4.00,1.063000,1.287500,1.470000


In [35]:
desc = entry[cols].describe()
print(desc.round(4).to_string())

med = entry[cols].median()
best = med.idxmax()
print(f"\nOptimal holding period: {best}  (median growth {med.max():.4f})")
print("\nmedian vs mean by month:")
summary = pd.DataFrame({"median": med, "mean": entry[cols].mean(),
                        "count": entry[cols].count()})
print(summary.round(4).to_string())

       future_growth_1_m  future_growth_2_m  future_growth_3_m  future_growth_4_m  future_growth_5_m  future_growth_6_m  future_growth_7_m  future_growth_8_m  future_growth_9_m  future_growth_10_m  future_growth_11_m  future_growth_12_m
count           123.0000           123.0000           123.0000           123.0000           123.0000           123.0000           123.0000           123.0000           123.0000            123.0000            123.0000            122.0000
mean              1.0685             1.1938             1.1681             1.0380             0.9723             1.0121             1.0469             0.9154             1.0822              1.1293              1.0035              0.9075
std               1.0537             1.7854             1.7923             1.3706             1.2342             1.5004             2.1965             2.0219             3.0191              3.4747              2.1355              2.1452
min               0.0729             0.1062         

In [36]:
# Question 4: [Strategy] Simple RSI-Based Trading Strategy - What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?

import sys
!{sys.executable} -m pip install gdown pyarrow

import gdown, pandas as pd, numpy as np

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 30.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [gdown]32m3/5 [filelock]

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=b1e67bdf-3228-4311-bef7-fb67f9ca89f3
To: /Users/osiosman/stock-analytics-zoomcamp/module2/data.parquet
100%|██████████| 130M/130M [00:06<00:00, 21.2MB/s] 


'data.parquet'

In [38]:
import pyarrow.parquet as pq
table = pq.read_table("data.parquet")
df = table.to_pandas()
print(df.shape)

(229932, 203)


In [39]:
print([c for c in df.columns if "rsi"    in c.lower()])
print([c for c in df.columns if "date"   in c.lower()])
print([c for c in df.columns if "growth_future" in c.lower()])

rsi_col  = "rsi"                 # adjust to what printed
date_col = "Date"                # adjust
fwd_col  = "growth_future_30d"

print(df[rsi_col].describe())
print(df[date_col].min(), "→", df[date_col].max())
print(df[fwd_col].describe())

['rsi', 'fastk_rsi', 'fastd_rsi', 'cdl3starsinsouth']
['Date']
['growth_future_30d']
count    229470.000000
mean         52.972321
std          12.085901
min           0.000000
25%          44.535042
50%          53.066628
75%          61.511647
max         100.000000
Name: rsi, dtype: float64
1972-06-01 00:00:00 → 2025-05-30 00:00:00
count    229767.000000
mean          1.004546
std           0.051762
min           0.238510
25%           0.979526
50%           1.003736
75%           1.028172
max           4.507647
Name: growth_future_30d, dtype: float64


In [40]:
if not np.issubdtype(df[date_col].dtype, np.datetime64):
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

selected_df = df[(df[rsi_col] < 30) &
                 (df[date_col] >= "2000-01-01") &
                 (df[date_col] <= "2025-06-01")]

print("signals:", len(selected_df))
print("with a 30d forward return:", selected_df[fwd_col].notna().sum())
print("missing forward return:", selected_df[fwd_col].isna().sum())

signals: 5206
with a 30d forward return: 5206
missing forward return: 0


In [41]:
net_income = 1000 * (selected_df[fwd_col] - 1).sum()
print(f"net income: ${net_income:,.2f}")
print(f"in $ thousands: {net_income/1000:,.2f}")

n = selected_df[fwd_col].notna().sum()
print(f"\ntrades: {n}")
print(f"avg per trade: ${net_income/n:,.2f}")
print(f"win rate: {(selected_df[fwd_col] > 1).sum() / n:.1%}")
print(f"capital deployed: ${1000*n:,.0f}")

net income: $65,805.59
in $ thousands: 65.81

trades: 5206
avg per trade: $12.64
win rate: 55.1%
capital deployed: $5,206,000
